# Medical Image Classification — HAM10000
This notebook demonstrates dataset loading, training (demo), evaluation and visualization. It is prepared to run in **Colab** or locally. See README.md for full instructions.

In [ ]:
# If running on Colab, uncomment and run the install block
# !pip install -q torch torchvision pandas matplotlib scikit-learn pillow tqdm seaborn
# If running locally, ensure requirements.txt is installed.

import os, sys
print('Python version:', sys.version)
print('Working dir:', os.getcwd())

import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
from torchvision.models import efficientnet_b0
import torch.nn as nn, torch.optim as optim

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## Dataset (HAM10000) — download options

Choose one of the following options and place images + metadata CSV under `data/` as described.

1. **Kaggle** (recommended):
   - HAM10000 on Kaggle: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
   - On Colab you can use the Kaggle API to download (requires uploading your `kaggle.json` credentials).

2. **ISIC Archive / Challenge**:
   - https://challenge.isic-archive.com/ — use the challenge download.

3. **Harvard Dataverse (DOI)**:
   - https://doi.org/10.7910/DVN/DBW86T


Required file layout (place into workspace):
```
data/
  raw/
    HAM10000_images_part_1/
    HAM10000_images_part_2/
  metadata.csv   # columns: image_path,label
```
Where `image_path` is relative path under `data/raw`, and `label` is integer class index (0..num_classes-1).

The notebook also includes a *small demo mode* that will run using a tiny random subset if the dataset is not present.


In [ ]:
# src/datasets/medical_images.py equivalent helpers in-notebook (lightweight)
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class MedicalImageDataset(Dataset):
    def __init__(self, csv_path, root_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.root = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root, row['image_path'])
        label = int(row['label'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


def get_transforms(img_size=224):
    return {
        'train': transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(0.1,0.1,0.1,0.02),
            transforms.ToTensor(),
        ]),
        'val': transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
    }

print('Helpers defined')


In [ ]:
# Attempt to find data/metadata.csv, otherwise create a tiny demo dataset using random images (placeholder)
DATA_DIR = 'data'
RAW_DIR = os.path.join(DATA_DIR, 'raw')
CSV_PATH = os.path.join(DATA_DIR, 'metadata.csv')

os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(CSV_PATH):
    print('Found metadata:', CSV_PATH)
    df = pd.read_csv(CSV_PATH)
else:
    print('metadata.csv not found — creating demo dataset (20 random images from torchvision).')
    # create demo csv using CIFAR10 images as placeholder
    demo_dir = os.path.join(RAW_DIR, 'demo')
    os.makedirs(demo_dir, exist_ok=True)
    from torchvision.datasets import CIFAR10
    ds = CIFAR10(root='./temp_cifar', download=True, train=True)
    demo_rows = []
    for i in range(20):
        img, lbl = ds[i]
        path = f'demo/img_{i}.png'
        img.save(os.path.join(RAW_DIR, path))
        demo_rows.append({'image_path': path, 'label': int(lbl % 7)})
    df = pd.DataFrame(demo_rows)
    df.to_csv(CSV_PATH, index=False)
    print('Saved demo metadata to', CSV_PATH)

# show first rows
df.head()


In [ ]:
# Build datasets and dataloaders
transforms = get_transforms(224)

# Split into train/val
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'] if 'label' in df.columns else None, random_state=42)
train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)

train_ds = MedicalImageDataset('data/train.csv', RAW_DIR, transform=transforms['train'])
val_ds   = MedicalImageDataset('data/val.csv', RAW_DIR, transform=transforms['val'])

train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=8)

print('Train size:', len(train_ds), 'Val size:', len(val_ds))

# show a batch of images
batch = next(iter(train_dl))
imgs, labels = batch
print('Batch images shape:', imgs.shape)

plt.figure(figsize=(8,4))
for i in range(min(8, imgs.shape[0])):
    plt.subplot(2,4,i+1)
    img = imgs[i].permute(1,2,0).numpy()
    plt.imshow((img - img.min())/(img.max()-img.min()))
    plt.title(str(labels[i].item()))
    plt.axis('off')
plt.tight_layout()


In [ ]:
# Build model (EfficientNet-B0) and quick demo training loop
num_classes = 7
model = efficientnet_b0(weights=None)
# If internet available or in Colab, set weights='IMAGENET1K_V1' to use pretrained
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# quick demo: 2 epochs over small demo dataset
EPOCHS = 2
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_dl, desc=f'Train epoch {epoch+1}'):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1} train loss:', running_loss/len(train_dl))

# Save demo checkpoint
os.makedirs('models/checkpoints', exist_ok=True)
torch.save(model.state_dict(), 'models/checkpoints/demo_best.pth')
print('Saved demo checkpoint to models/checkpoints/demo_best.pth')


In [ ]:
# Evaluation on validation set
model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for images, labels in val_dl:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_targets.extend(labels.numpy().tolist())

print(classification_report(all_targets, all_preds, zero_division=0))

# Confusion matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion matrix (demo)')
plt.show()


In [ ]:
# Inference on a single image
from PIL import Image
sample_img_path = val_df.iloc[0]['image_path']
sample_img = Image.open(os.path.join(RAW_DIR, sample_img_path)).convert('RGB')
plt.imshow(sample_img); plt.axis('off'); plt.title('Sample image')

# preprocess and predict
img_t = transforms['val'](sample_img).unsqueeze(0).to(DEVICE)
model.eval()
with torch.no_grad():
    out = model(img_t)
    probs = torch.softmax(out, dim=1).cpu().numpy()[0]
print('Pred probs:', probs)
print('Pred label:', int(probs.argmax()))


---
## Colab-specific instructions
1. Upload your `kaggle.json` (from Kaggle Account -> API) to Colab and run:
```python
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data --unzip
```
2. After download, move images and metadata so that `data/raw/` contains images and `data/metadata.csv` exists.
3. Run the notebook cells.

If you prefer ISIC, follow ISIC instructions to obtain images and metadata and place them under `data/`.
---
